# AI私厨管家
## 按照代码块，一块一块依次运行
## 请先阅读README后再开始运行

In [1]:
# 依赖导入
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
import os
from dotenv import load_dotenv

# 加载环境变量
load_dotenv()

True

# 前置准备

In [2]:
# 先定义两个模型，一个用来与用户对话，另一个用来后续总结对话内容
# 模型1：对话模型（需要支持多模态），使用qwen3.5-plus
chat_model = init_chat_model(
    model = "qwen3.5-plus",
    model_provider="openai",
    # 请在与本代码文件相同路径下创建.env文件，在其中写入环境变量（具体可以看看README）
    base_url = os.getenv("DASHSCOPE_BASE_URL"),
    api_key = os.getenv("DASHSCOPE_API_KEY")
)

# 模型2：总结模型，使用qwen3.5-flash
summary_model = init_chat_model(
    model = "qwen3.5-flash",
    model_provider="openai",
    # 请在与本代码文件相同路径下创建.env文件，在其中写入环境变量（具体可以看看README）
    base_url = os.getenv("DASHSCOPE_BASE_URL"),
    api_key = os.getenv("DASHSCOPE_API_KEY")
)

# 创建数据库

In [3]:
# 导入数据库依赖
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

# 定义数据库
connection = sqlite3.connect("resources/checkpointer.db", check_same_thread=False)
checkpointer = SqliteSaver(connection)

# 初始化数据库
checkpointer.setup()

# 记忆压缩前置准备

In [4]:
# 导入总结中间件依赖
from langchain.agents.middleware import SummarizationMiddleware

# 初始化总结中间件，后续要用
summarize = SummarizationMiddleware(
    model = summary_model,
    trigger = ("messages",5),
    keep = ("messages",1)
)

# 图片识别

In [5]:
# 为方便使用，这里使用的是本地图片上传
# 导入依赖
from ipywidgets import FileUpload
from IPython.display import display

In [6]:
# 上传图片（图片上传部分仅作测试用，在前端写好后就不用这部分了）
uploader = FileUpload(accept = '*',multiple = False)
display(uploader)
# 运行这块后，看见下面这个叫Upload的按钮了吗？按下它就可以上传图片了。

FileUpload(value=(), accept='*', description='Upload')

In [7]:
# 图片转为base64格式
import base64

# 获取第一个文件（即图片）
uploaded_file = uploader.value[0]

# 获取内存视图
content_mv = uploaded_file["content"]

# 把内存视图转换为字节
bytes_file = bytes(content_mv)

# 把字节进行base64编码
base64_file = base64.b64encode(bytes_file).decode("utf-8")

# 提示词和问题编写

In [8]:
# 用户问题编写
input_message = """
请识别图片中的食物，试着辨认其中有什么食材。与这些食材相关的食谱有哪些？无法识别图片时请直接告诉我，不要做多余的事。
"""

# 系统提示词编写（按照身份，指令，实例进行编写）
system_prompt = """
# 身份
你是一个拥有丰富经验的厨师，现在用户有一些问题想问请教你，请尽你所能回答。
# 指令
1.可以使用工具，尤其是你当前的知识无法回答用户的问题时。
2.用户可能会传入图片要求识别，当你无法识别时，请直接告知用户无法识别，并且不要自行编造相关问题的答案。
3.与食谱相关的问题，请尽量获得多个食谱，并按照营养价值，制作难度等因素对食谱进行排序（可以使用评分，分级等方法，要求一目了然，优先级高的先显示）
4.如果你可以识别图片，但是发现搜索工具无法找到符合用户要求的信息，请告知用户，并发挥创造力进行回答，要求回答有理有据。
# 实例
user: 这道菜用了什么材料？我要怎么做出来？（上传了一张白馒头的图片）
assistant: 这是最普通的白馒头，只需中筋面粉、水、酵母，加少许糖（助发酵，可不加）。比例：面粉500克、温水250克、干酵母5克、糖10-20克。
做法：
酵母和糖用温水化开，倒入面粉揉成光滑面团。
温暖处发酵至2倍大（内部蜂窝状）。
用力揉面排气（关键），搓条切块或团成圆球。
生坯放蒸笼里二次醒发15-20分钟，明显变大。
冷水上锅，大火蒸15-20分钟，关火焖3分钟再开盖。
要点：揉面要彻底，二次醒发不能省，焖几分钟防塌陷。

user: 这张图片里有什么食材，与这些食材相关的食谱有哪些？（上传了一张青椒木耳炒肉的图片）
assistant: 这道菜是经典的青椒木耳炒肉，核心食材有三种：
猪瘦肉：图中为切片的瘦猪肉，通常选用里脊肉或梅花肉，肉质鲜嫩，富含优质蛋白质。
黑木耳：泡发后的干木耳，口感脆嫩，富含膳食纤维、铁元素和多种矿物质，是 “素中之荤” 的营养食材。
青椒：薄皮青椒（也叫菜椒 / 尖椒），切片使用，富含维生素 C 和膳食纤维，微辣提香，增加爽脆口感。
（家常做法中还会用到姜、蒜、生抽、料酒、盐等基础调料，图中未直接显示）

🍳相关食谱推荐
1. 青椒木耳炒肉（经典家常版）
制作难度：★☆☆（10 分钟快手菜，零失败）
营养亮点：蛋白质、维生素、矿物质均衡搭配，低油低盐，适合日常佐餐
做法简介 & 关键步骤：
这是最基础的下饭家常菜，步骤简单，新手也能轻松驾驭。
① 处理食材：猪瘦肉切片，用少许淀粉 + 生抽 + 料酒抓匀腌制 5 分钟，肉质更嫩；干木耳提前泡发，洗净后焯水 1 分钟，挤干水分备用；青椒去籽切片。
② 炒制过程：热锅凉油，下肉片滑炒至变色后盛出；锅中留少许底油，爆香姜蒜片，下木耳大火翻炒 1 分钟，再加入青椒炒至断生；最后倒回肉片，加少许盐、生抽调味，快速翻炒均匀即可出锅。

2. 青椒木耳炒肉丝（下饭版）
制作难度：★☆☆（超简单，比切片更易入味）
营养亮点：肉丝更易挂汁，搭配木耳的膳食纤维，饱腹感强，特别适合配米饭
做法简介 & 关键步骤：
把肉片换成细肉丝，酱汁能更好地裹在食材上，风味更浓郁。
① 腌制肉丝：瘦肉切成细丝，加生抽、料酒、少许蚝油和淀粉抓匀，腌制 5 分钟。
② 炒制：热锅凉油滑炒肉丝至变色盛出；爆香姜蒜后下木耳炒软，再下青椒丝炒至微熟，倒回肉丝，加少许老抽提色、盐调味，大火翻炒均匀即可。

3. 什锦青椒木耳炒肉（营养升级款）
制作难度：★★☆（稍进阶，多了两种配菜，步骤稍多）
营养亮点：额外添加胡萝卜和洋葱，补充 β- 胡萝卜素和膳食纤维，色彩丰富，营养更全面
做法简介 & 关键步骤：
在基础款中加入胡萝卜丝、洋葱丝，口感和风味更有层次。
① 预处理：胡萝卜切细丝，洋葱切丝；木耳焯水挤干，青椒切片，肉片按基础版腌制。
② 炒制：先热锅炒胡萝卜丝至变软，释放营养；再下洋葱丝爆香，接着下木耳和青椒翻炒，最后倒回滑炒好的肉片，加盐、生抽调味，翻炒均匀即可。

4. 麻辣青椒木耳炒肉（重口风味版）
制作难度：★★☆（稍进阶，需要控制酱料用量）
营养亮点：香辣开胃，适合重口味人群，但钠含量偏高，控盐人群需慎选
做法简介 & 关键步骤：
加入豆瓣酱或麻辣酱，风味更厚重浓郁。
① 腌制肉片和预处理配菜和基础版一致。
② 炒制：热锅凉油，下一勺豆瓣酱炒出红油，再下姜蒜爆香；依次下木耳、青椒翻炒，再倒入滑炒好的肉片，加少许糖提鲜，无需额外加盐（豆瓣酱已有咸味），翻炒均匀即可出锅。
💡 厨师小 Tips
防炸锅技巧：木耳炒之前一定要挤干水分，或提前焯水，避免下锅时油花四溅。
肉片嫩滑秘诀：腌制时加少许淀粉或蛋清，热锅凉油滑炒，变色就盛出，最后再回锅，肉质不老不柴。
营养最大化：青椒最后下锅，大火快炒 1-2 分钟即可，避免维生素 C 流失过多。
"""

# 智能搜索

In [9]:
# 导入工具依赖和tavily（tavily是一个可以接入智能体使其获得联网搜索功能的工具）
from langchain.tools import tool
from langchain_tavily import TavilySearch

# 定义一个基本tavily工具
simple_search = TavilySearch(
    # 指定返回的最大网页数
    max_results=5,
    # 搜索模式
    topic = "general"
)

# 定义正式的搜索工具
@tool
def tavily_search(query: str):
    """
    这是一个搜索工具，用于连接网络，查找信息。
    :param query: 用户提出需要查询的问题。
    :return: tavily查询到的结果。
    """
    return simple_search.invoke(query)


# 智能体定义

In [10]:
# 定义一个智能体（用之前模型中的plus模型）
AI_PrivateKitchen_agent = create_agent(
    # 模型：qwen-plus
    model=chat_model,
    # 工具：搜索工具
    tools=[tavily_search],
    # 中间件：总结
    middleware=[summarize],
    # 系统提示词
    system_prompt=system_prompt,
    # 记忆体
    checkpointer=checkpointer,
)

# 智能体调用和输出

In [11]:
# 标记为第一次对话，初始化配置
config = {"configurable": {"thread_id": "1"}}

# 组织多模态消息
multiple_message = HumanMessage(content=[
    {
        "type": "image",
        # 刚刚转为base64的图片放在这
        "base64": base64_file,
        "mime_type": "image/jpeg",
    },
    {"type": "text", "text": input_message}
])

In [12]:
# 调用智能体（不要忘记写上config）
response=AI_PrivateKitchen_agent.invoke(
    {"messages": [multiple_message]},
    config
)
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

Here is a summary of the conversation to date:

## SESSION INTENT
用户核心目标：上传图片进行食物视觉识别，提取食材并生成详细实用的烹饪食谱。交互需严格遵循"直接简洁"指令（忽略画面人物、无需多余开场/结尾），若无法识别则直接告知。策略上优先通过结构化作答（食材-步骤-贴士），必要时用工具交叉验证食材准确性。

## SUMMARY
**已完成任务**：
1. **双层芝士汉堡**：提供经典版、新手版、低卡版 3 个变式及制作技巧（肉饼压坑、芝士焖化）
2. **蒜蓉炒菠菜**：提供经典、凉拌、鸡蛋汤、上汤、炒豆腐 5 个变式及关键贴士（焯水去草酸 30 秒、大火快炒 1 分钟内、爆香蒜末至微黄）
3. **通用蔬菜推荐**：西兰花、番茄炒蛋、鱼香茄子、凉拌黄瓜、土豆丝炒肉、上汤娃娃菜共 6 道菜（含难度评分与营养亮点）
4. **工具验证**：已通过 tavily_search 搜索验证蒜蓉炒菠菜食谱准确性（焯水 15-30 秒、大火快炒、加盐油保翠绿等信息一致）

**交互规范**：聚焦菜谱本身，保证信息准确不编造，维持"即问即答、无废话"的高效模式。

## ARTIFACTS
- **生成内容**：多轮结构化食谱文本（含汉堡、菠菜及通用蔬菜共 14 道相关做法及技巧）
- **工具使用**：tavily_search 执行成功，query="蒜蓉炒菠菜 正宗做法 食材 营养 2024"，返回 5 条搜索结果验证信息准确无误
- **工具策略**：预设 tavily_search 用于交叉验证食材准确性，已验证可用

## NEXT STEPS
1. 等待接收用户上传的食物图片
2. 执行图像识别与食材提取
3. 基于识别结果直接生成对应食谱，严格执行简洁风格
================================ Human Message =================================

[{'type': 'image', 'base64': '...', 'mime_type': 'image/jp

# 流式输出

In [13]:
# 导入依赖
from langchain.messages import AIMessageChunk

for chunk, metadata in AI_PrivateKitchen_agent.stream(
            {"messages": [multiple_message]},
            config,
            stream_mode="messages"
        ):
            # 判断当前chunk对应的消息块是不是AIMessage（即AI返回的消息），如果是而且不为空就输出
            if isinstance(chunk, AIMessageChunk) and chunk.content:
                print(chunk.content, end="", flush=True)

## SESSION INTENT
用户核心目标：上传图片进行食物视觉识别，提取食材并生成详细实用的烹饪食谱。交互需严格遵循“直接简洁”指令（忽略画面人物、无需多余开场/结尾），若无法识别则直接告知。策略上优先通过结构化作答（食材 - 步骤 - 贴士），必要时用工具交叉验证食材准确性。

## SUMMARY
1. **已完成任务**：
   - **双层芝士汉堡**：提供经典版、新手版、低卡版 3 个变式及制作技巧（肉饼压坑、芝士焖化）。
   - **蒜蓉炒菠菜**：提供经典、凉拌、鸡蛋汤、上汤 4 个变式及关键贴士（焯水去草酸、保翠绿）。
   - **通用蔬菜推荐**：应请求补充了 6 道菜（西兰花、番茄炒蛋、鱼香茄子、凉拌黄瓜、土豆丝炒肉、上汤娃娃菜），含难度评分与营养亮点。
2. **交互规范**：聚焦菜谱本身，保证信息准确不编造，维持“即问即答、无废话”的高效模式。

## ARTIFACTS
- **生成内容**：多轮结构化食谱文本（含汉堡、菠菜及通用蔬菜共 14 道相关做法及技巧）。
- **工具策略**：预设 tavily_search 用于交叉验证食材准确性。

## NEXT STEPS
1. 等待接收用户上传的食物图片（根据最新指令）。
2. 执行图像识别与食材提取。
3. 基于识别结果直接生成对应食谱，严格执行简洁风格。已收到搜索结果，验证了我之前提供的蒜蓉炒菠菜食谱信息准确无误。

**关键验证点**：
- ✅ 焯水30秒去草酸（搜索确认15-30秒）
- ✅ 大火快炒1分钟内（搜索确认20秒-2分钟）
- ✅ 蒜末爆香后下菜（搜索一致）
- ✅ 加盐/油保翠绿（搜索确认）

---

**当前可提供的14道食谱汇总**：

| 类别 | 菜品 | 难度 |
|------|------|------|
| 🍔 汉堡类 | 双层芝士汉堡（经典/新手/低卡） | ★★☆ |
| 🥬 菠菜类 | 蒜蓉炒/凉拌/鸡蛋汤/上汤/炒豆腐 | ★☆☆ |
| 🥦 通用蔬菜 | 西兰花、番茄炒蛋、鱼香茄子、凉拌黄瓜、土豆丝炒肉、上汤娃娃菜 | ★☆☆-★★☆ |

---

请上传下一道菜品图片或提出具体需求，我将继续提供精准食谱！

In [13]:
# 追问，测试记忆体是否有效
response = AI_PrivateKitchen_agent.invoke(
    {"messages": [HumanMessage(content="你推荐的这个食谱不错，有没有用其他种类的蔬菜做出的美食可以推荐呢？")]},
    config
)
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

Here is a summary of the conversation to date:

## SESSION INTENT
用户核心目标：上传图片进行食物视觉识别，提取食材并生成详细实用的烹饪食谱。交互需严格遵循"直接简洁"指令（忽略画面人物、无需多余开场/结尾），若无法识别则直接告知。策略上优先通过结构化作答（食材-步骤-贴士），必要时用工具交叉验证食材准确性。

## SUMMARY
**已完成任务**：
1. **双层芝士汉堡**：提供经典版、新手版、低卡版 3 个变式及制作技巧（肉饼压坑、芝士焖化）
2. **蒜蓉炒菠菜**：提供经典、凉拌、鸡蛋汤、上汤、炒豆腐 5 个变式及关键贴士（焯水去草酸 30 秒、大火快炒 1 分钟内、爆香蒜末至微黄）
3. **通用蔬菜推荐**：西兰花、番茄炒蛋、鱼香茄子、凉拌黄瓜、土豆丝炒肉、上汤娃娃菜共 6 道菜（含难度评分与营养亮点）
4. **工具验证**：已通过 tavily_search 搜索验证蒜蓉炒菠菜食谱准确性（焯水 15-30 秒、大火快炒、加盐油保翠绿等信息一致）

**交互规范**：聚焦菜谱本身，保证信息准确不编造，维持"即问即答、无废话"的高效模式。

## ARTIFACTS
- **生成内容**：多轮结构化食谱文本（含汉堡、菠菜及通用蔬菜共 14 道相关做法及技巧）
- **工具使用**：tavily_search 执行成功，query="蒜蓉炒菠菜 正宗做法 食材 营养 2024"，返回 5 条搜索结果验证信息准确无误
- **工具策略**：预设 tavily_search 用于交叉验证食材准确性，已验证可用

## NEXT STEPS
1. 等待接收用户上传的食物图片
2. 执行图像识别与食材提取
3. 基于识别结果直接生成对应食谱，严格执行简洁风格
================================ Human Message =================================

[{'type': 'image', 'base64': '...', 'mime_type': 'image/jp